In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تثبيت إضافات المُحوِّل واستخدامها

<details>
<summary><b>إصدارات الحزم</b></summary>

الكود في هذه الصفحة جرى تطويره باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو ما هو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

لتسهيل تطوير كود التحويل (transpilation) المخصص وإعادة استخدامه من قِبَل مجتمع مستخدمي Qiskit الأوسع، يدعم Qiskit SDK واجهة إضافات (plugin interface) تتيح لحزم Python الخارجية الإعلان عن تقديمها وظائف تحويل موسّعة يمكن الوصول إليها عبر Qiskit.

حاليًا، يمكن للإضافات الخارجية توفير وظائف التحويل الموسّعة بثلاث طرق:

- [إضافة مرحلة Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins) توفّر مدير تمريرات (pass manager) يمكن استخدامه بدلًا من إحدى [المراحل الست](transpiler-stages) في مدير التمريرات المرحلي المُعدّ مسبقًا: `init` و`layout` و`routing` و`translation` و`optimization` و`scheduling`.
- [إضافة تركيب أحادي الوحدة (unitary synthesis)](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) توفّر وظائف موسّعة لتركيب البوابات الأحادية.
- [إضافة التركيب عالي المستوى (high-level synthesis)](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) توفّر وظائف موسّعة لتركيب "الكائنات عالية المستوى" كالدوال الخطية أو معاملات Clifford. تُمثَّل الكائنات عالية المستوى بالفئات الفرعية من فئة [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

يصف الجزء التالي من الصفحة كيفية عرض الإضافات المتاحة وتثبيت إضافات جديدة واستخدامها.

## عرض الإضافات المتاحة وتثبيت إضافات جديدة
يتضمّن Qiskit بالفعل بعض الإضافات المدمجة للتحويل. لتثبيت المزيد، يمكنك استخدام مدير حزم Python الخاص بك. مثلًا، يمكنك تشغيل `pip install qiskit-toqm` لتثبيت إضافة مرحلة التوجيه [Qiskit TOQM](https://github.com/qiskit-toqm/qiskit-toqm). عدد من الإضافات الخارجية هي جزء من [نظام Qiskit البيئي](https://qiskit.github.io/ecosystem/#transpiler_plugin).

### عرض إضافات مرحلة Transpiler المتاحة
استخدم دالة [list_stage_plugins](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#qiskit.transpiler.preset_passmanagers.plugin.list_stage_plugins)، مع تمرير اسم المرحلة التي تريد عرض إضافاتها.

In [1]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

In [2]:
list_stage_plugins("routing")

['basic', 'default', 'lookahead', 'none', 'sabre']

 If `qiskit-toqm` were installed, then `toqm` would appear in the list of `routing` plugins.

### List available unitary synthesis plugins

Use the [unitary_synthesis_plugin_names](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.unitary_synthesis_plugin_names) function.

In [3]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

### List available high-level synthesis plugins

Use the [high_level_synthesis_plugin_names](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.high_level_synthesis_plugin_names) function, passing the name of the type of "high-level object" to be synthesized. The name corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the [Operation](/docs/api/qiskit/qiskit.circuit.Operation) class representing the type of object being synthesized.

In [4]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

لو كان `qiskit-toqm` مثبَّتًا، لظهر `toqm` في قائمة إضافات `routing`.

### عرض إضافات التركيب الأحادي (unitary synthesis) المتاحة
استخدم دالة [unitary_synthesis_plugin_names](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.unitary_synthesis_plugin_names).

In [5]:
from qiskit.transpiler.passes.synthesis.plugin import (
    HighLevelSynthesisPluginManager,
)

HighLevelSynthesisPluginManager().plugins.names()

['FullAdder.default',
 'FullAdder.ripple_c04',
 'FullAdder.ripple_v95',
 'HalfAdder.default',
 'HalfAdder.qft_d00',
 'HalfAdder.ripple_c04',
 'HalfAdder.ripple_r25',
 'HalfAdder.ripple_v95',
 'IntComp.default',
 'IntComp.noaux',
 'IntComp.twos',
 'ModularAdder.default',
 'ModularAdder.modular_v17',
 'ModularAdder.qft_d00',
 'ModularAdder.ripple_c04',
 'ModularAdder.ripple_v95',
 'Multiplier.cumulative_h18',
 'Multiplier.default',
 'Multiplier.qft_r17',
 'PauliEvolution.default',
 'PauliEvolution.rustiq',
 'WeightedSum.default',
 'annotated.default',
 'clifford.ag',
 'clifford.bm',
 'clifford.default',
 'clifford.greedy',
 'clifford.layers',
 'clifford.lnn',
 'linear_function.default',
 'linear_function.kms',
 'linear_function.pmh',
 'mcmt.default',
 'mcmt.noaux',
 'mcmt.vchain',
 'mcmt.xgate',
 'mcx.1_clean_b95',
 'mcx.1_clean_kg24',
 'mcx.1_dirty_kg24',
 'mcx.2_clean_kg24',
 'mcx.2_dirty_kg24',
 'mcx.default',
 'mcx.gray_code',
 'mcx.n_clean_m15',
 'mcx.n_dirty_i15',
 'mcx.noaux_hp24'

## Use a plugin

In this section, we show how to use transpiler plugins. In the code examples, we use plugins that come with Qiskit, but plugins installed from third-party packages are used the same way.

### Use a transpiler stage plugin

To use a transpiler stage plugin, specify its name with the appropriate argument to [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). The argument is formed by appending `_method` to the name of the transpilation stage. For example, to use the `lookahead` routing plugin, we would specify `lookahead` for the `routing_method` argument:

In [6]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, routing_method="lookahead"
)

### عرض إضافات التركيب عالي المستوى (high-level synthesis) المتاحة
استخدم دالة [high_level_synthesis_plugin_names](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.high_level_synthesis_plugin_names)، مع تمرير اسم نوع "الكائن عالي المستوى" المراد تركيبه. يتوافق الاسم مع الخاصية [`name`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation#name) في فئة [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) التي تُمثّل نوع الكائن الجاري تركيبه.

In [7]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    unitary_synthesis_method="sk",
    unitary_synthesis_plugin_config=dict(
        basis_gates=["cz", "id", "rz", "sx", "x"]
    ),
)

Unitary synthesis is used in the `init`, `translation`, and `optimization` stages of the staged pass manager returned by [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or used in [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). See [Transpiler stages](transpiler-stages) for a description of these stages.

Use the `unitary_synthesis_plugin_config` argument, a free-form dictionary, to pass options for the unitary synthesis method. The documentation of the synthesis method should explain the options it supports. See [this list](/docs/api/qiskit/transpiler_synthesis_plugins#unitary-synthesis-plugins-2) for links to the documentation of the built-in unitary synthesis plugins.

### Use a high-level synthesis plugin

First, create an [HLSConfig](/docs/api/qiskit/qiskit.transpiler.passes.HLSConfig) to
store the names of the plugins to use for various high-level objects. For example:

In [8]:
from qiskit.transpiler.passes import HLSConfig

hls_config = HLSConfig(clifford=["layers"], linear_function=["pmh"])

يمكنك استخدام فئة [HighLevelSynthesisPluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPluginManager) لعرض أسماء جميع إضافات التركيب عالي المستوى:

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, hls_config=hls_config
)

High-level synthesis is used in the `init`, `translation`, and `optimization` stages of the staged pass manager returned by [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or used in [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). See [Transpiler stages](transpiler-stages) for a description of these stages.

## Next steps

<Admonition type="tip" title="Recommendation">
    - [Create a transpiler plugin](./create-transpiler-plugin).
    - Check out the [tutorials](/docs/tutorials) for examples of transpiling and running quantum circuits.
</Admonition>